In [1]:
# LVDT_simulation/tests/demo.ipynb
import sys
sys.path.append('../../')
import time
import datetime
import random
import numpy as np
from core.simulator import SimulatorManager
from core.evaluator import lvdt_evaluator, vc_evaluator
from joblib import Parallel, delayed
import multiprocessing as mp


#### functions


In [ ]:
def run_femm_lvdt(process_id, params):
    print(f"Process {process_id} started")
    process_id = str(process_id) + f"_{random.randint(100, 999)}"

    f1, f2 = lvdt_evaluator(
        params = params,
        input_json_filename='../config//v0_typeG_lvdt.json',
        iter_json_filename=f'../tests/iteration_config_lvdt_{process_id}.json',
        output_dir="../tests/",
        output_filename=f'lvdt_output_{process_id}.h5')
    return f1, f2

def run_femm_vc(process_id, params):
    print(f"Process {process_id} started")
    time = datetime.datetime.now().strftime("%Y%m%d%H%M%S") + f"_{random.randint(100, 999)}" + f"_{process_id}"

    f3, f4 = vc_evaluator(
        params = params,
        input_json_filename='../config/v0_typeG_vc.json',
        iter_json_filename=f'../tests/iteration_config_vc_{time}.json',
        output_dir="../tests/",
        output_filename=f'vc_output_{time}.h5')
    return f3, f4

def run_simulation(process_id, params):
    f1, f2 = run_femm_lvdt(process_id, params)

    f3, f4 = run_femm_vc(process_id, params)

    return f1, f2, f3, f4


In [ ]:
n_processes = 4
pool = mp.Pool(processes=n_processes)

x = [6,14,20,7,34,25,17.5]
X = np.array([x]*n_processes)
t_start = time.time()
results = pool.starmap(run_simulation, [(i, {}) for i in range(n_processes)])
t_end = time.time()


In [ ]:
n_job  = 10
x = [6, 14, 20, 7, 34, 25, 17.5]
X = np.array([x]*n_job)
t_s = time.time()
result = Parallel(n_jobs=n_job)(delayed(run_simulation)(i, x) for i in range(n_job))
t_e = time.time()
print(f"Total time: {t_e - t_s}")   


#### LVDT test

In [ ]:
n_job  = 10
x = [6, 14, 20, 7, 34, 25, 17.5]
X = np.array([x]*n_job)
t_s = time.time()
result = Parallel(n_jobs=n_job)(delayed(run_femm_lvdt)(i, x) for i in range(n_job))
t_e = time.time()
print(f"Total time: {t_e - t_s}")   


Total time: 253.86664009094238


#### VC test


In [5]:
n_job  = 2
x = [6, 14, 20, 7, 34, 25, 17.5]
X = np.array([x]*n_job)
t_s = time.time()
result = Parallel(n_jobs=n_job, backend='loky')(delayed(run_femm_vc)(i, x) for i in range(n_job))
t_e = time.time()
print(f"Total time: {t_e - t_s}")   


Total time: 34.897520780563354
